# Load files

## Input files format

## The different objects

## Raw files

### Example

In [ ]:
from multipac_testbench import PowerStep
from multipac_testbench.data import config_path
from multipac_testbench.data.power_steps import power_step_example
from multipac_testbench.instruments import FieldProbe

We load a trigger raw data file, without declaring it as raw.
All the signals are between 0 and 10 because they are acquisition voltages, not actual physical quantity.

In [ ]:
raw = PowerStep(power_step_example, config_path, freq_mhz=140.0, swr=4.0, sample_index=1)
_ = raw.sweet_plot(FieldProbe, figsize=(8, 4))

In [ ]:
treated_raw = PowerStep(power_step_example, config_path, freq_mhz=140.0, swr=4.0, sample_index=1, is_raw=True, create_virtual_instruments=False)
_ = treated_raw.sweet_plot(FieldProbe, figsize=(8, 4))

## Trigger policy

### Example

In [ ]:
from multipac_testbench import MultipactorTest
from multipac_testbench.data import config_path
from multipac_testbench.data.multipactor_tests import test_120MHz_SWR1_4
from multipac_testbench.instruments import CurrentProbe, FieldProbe

to_plot = CurrentProbe, FieldProbe

keep_all = MultipactorTest(test_120MHz_SWR1_4, config_path, freq_mhz=120.0, swr=1.0, info="Keep all triggers", is_raw=True)
_ = keep_all.sweet_plot(*to_plot, figsize=(8, 8))

average =  MultipactorTest(test_120MHz_SWR1_4, config_path, freq_mhz=120.0, swr=1.0, info="Average triggers", trigger_policy="average", is_raw=True)
_ = average.sweet_plot(*to_plot, figsize=(8, 8))

## Concatenation of power step files

### Example

In [ ]:
from functools import partial
from pathlib import Path
import tempfile

from numpy.typing import NDArray

from multipac_testbench import MultipactorTest, PowerStepSet
from multipac_testbench.data import config_path
from multipac_testbench.data.power_steps import power_step_set_example
from multipac_testbench.multipactor_test.helper import take_median

power_step_set = PowerStepSet(
    power_step_set_example,
    config_path,
    freq_mhz=140.0,
    swr=1.0,
    are_raw=False,  # recommended, in order to keep the output file raw
    create_virtual_instruments=False,
)

In [ ]:
def average(raw_data: NDArray) -> float:
    """Return average over last 100 points."""
    return take_median(raw_data, first_index=-100, last_index=-1)

def average_for_power(raw_data: NDArray) -> float:
    """Return average over last 40 points.
    
    Use it for Power signals, which are shifted wrt other signals.
    
    """
    return take_median(raw_data, first_index=-40, last_index=-1)


#  In the real life, provide a real CSV path, not a tmp one
with tempfile.NamedTemporaryFile(suffix=".csv") as tmp:
    csv_path = Path(tmp.name)
    power_step_set.to_multipactor_test_file(
    csv_path=csv_path,
    reducer=average,
    special_reducers={
        "NI9205_Power1": average_for_power,
        "NI9205_Power2": average_for_power
    },
    )

    # Re-use it
    multipactor_test = MultipactorTest(csv_path, config_path, freq_mhz=140.0, swr=1.0, is_raw=True)